# 04 - Train MLP
Đánh giá MLP PyTorch thật; log theo từng fold được ghi vào file log.


## Chuẩn bị


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT = PROJECT_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)



## Hàm: `log`


In [2]:
def log(message):
    print(message)
    with (OUTPUT / "train_log.txt").open("a", encoding="utf-8") as file:
        file.write(f"{message}\n")

import pandas as pd
from notebook_loader import run_notebook_code
from lab03_plots import plot_fold_scores
run_notebook_code(PROJECT_ROOT / "house_price_pipeline.ipynb", globals())
run_notebook_code(PROJECT_ROOT / "pytorch_mlp.ipynb", globals())



## Hàm: `mlp_cv`


In [3]:
def mlp_cv(train: pd.DataFrame, folds: int, random_state: int, quick: bool):
    # Cross-validation riêng cho MLP PyTorch nhưng vẫn dùng tiền xử lý của sklearn.
    x = make_feature_frame(train, "engineered")
    y = np.log1p(train[TARGET].astype(float).to_numpy())
    kf = KFold(n_splits=folds, shuffle=True, random_state=random_state)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(x), start=1):
        estimator = Pipeline(
            [
                ("preprocess", make_preprocessor(x.iloc[tr_idx])),
                (
                    "model",
                    TorchMLPRegressor(
                        max_epochs=90 if quick else 280,
                        patience=14 if quick else 28,
                        random_state=random_state + fold,
                    ),
                ),
            ]
        )
        estimator.fit(x.iloc[tr_idx], y[tr_idx])
        pred = estimator.predict(x.iloc[va_idx])
        score = kaggle_log_rmse(y[va_idx], pred)
        print(f"MLP fold {fold}/{folds}: log-RMSE={score:.6f}")
        scores.append(score)

    return {
        "experiment": "engineered_mlp",
        "feature_profile": "engineered",
        "model": "pytorch_mlp",
        "remove_outliers": False,
        "rows_used": int(len(train)),
        "rows_removed": 0,
        "cv_folds": int(folds),
        "log_rmse_mean": float(np.mean(scores)),
        "log_rmse_std": float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "fold_scores": [float(s) for s in scores],
    }





## Chạy notebook


In [4]:
log("[5] Training MLP...")
train, _ = load_competition_data(PROJECT_ROOT / "data")
result = mlp_cv(train, folds=3, random_state=42, quick=True)
pd.DataFrame([result]).to_csv(OUTPUT / "mlp_results.csv", index=False)
log(f"MLP CV log-RMSE={result['log_rmse_mean']:.6f}")
plot_fold_scores(result["fold_scores"], OUTPUT / "mlp_fold_scores.png", "MLP cross-validation by fold")


[5] Training MLP...


c:\ProgramData\anaconda3\Lib\site-packages\torch\autograd\graph.py:1072: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


MLP fold 1/3: log-RMSE=1.245535
MLP fold 2/3: log-RMSE=0.932487
MLP fold 3/3: log-RMSE=0.895700
MLP CV log-RMSE=1.024574
